# Notebook 12 — Pilot DFT job packaging and reaction-balance audit

**Private execution notebook.** This notebook prepares the first exact-state pilot DFT package. It does not run VASP.

## Pilot case

The locked first pilot case is:

```text
Na_candidate_06
charged:    NaCoPCO7
discharged: Na3CoPCO7
```

The exact charged/discharged structures and endpoint IDs come from Notebook 11.

## What this notebook does

1. verifies the complete Notebook 11 package and its SHA256 manifest;
2. loads the exact charged and discharged structures;
3. calculates cell formula-unit multiplicities and balances the Na insertion reaction;
4. writes a machine-readable voltage and volume-change formula;
5. prepares a bcc metallic-Na reference input;
6. packages charged, discharged and Na-reference jobs;
7. creates scheduler scripts and a strict execution checklist;
8. creates a parser-ready result manifest for the later DFT-results notebook;
9. checks that no licensed `POTCAR` file is present.

## Inputs

Required:

```text
canonical Notebook 11 repository artifacts
```

Optional:

```text
pilot_dft_resource_config.yaml
```

Expected final decision without a confirmed resource file:

```text
PILOT_PACKAGE_READY_RESOURCE_CONFIRMATION_REQUIRED
```

Expected final decision with a complete, confirmed resource file:

```text
FULL_GO_SUBMIT_PILOT_DFT_JOBS
```

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import shutil
import stat
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yaml

try:
    from pymatgen.core import Composition, Lattice, Structure
    from pymatgen.io.vasp import Kpoints, Poscar
    from pymatgen.io.vasp.sets import MPRelaxSet, MPStaticSet
    import pymatgen
except Exception as exc:
    raise ImportError(
        "Notebook 12 requires pymatgen. Install it with: "
        "%pip install pymatgen monty pyyaml"
    ) from exc

def _locate_repository_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for root in [candidate, *candidate.parents]:
        if (
            (root / "notebooks").is_dir()
            and (root / "data").is_dir()
            and (root / "results").is_dir()
            and (root / "provenance").is_dir()
        ):
            return root
    raise FileNotFoundError("Could not locate the clean-room repository root.")


REPOSITORY_ROOT = _locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from software.cmt_repository_paths import artifact_namespace, runtime_cache_root

ROOT = REPOSITORY_ROOT
OUTPUT_ROOT = artifact_namespace("12", REPOSITORY_ROOT)
AUDIT_DIR = OUTPUT_ROOT / "audit"
PROCESSED_DIR = OUTPUT_ROOT / "processed"
METADATA_DIR = OUTPUT_ROOT / "metadata"
PILOT_DIR = OUTPUT_ROOT / "pilot_dft_package"
LOG_DIR = OUTPUT_ROOT / "logs"
CACHE_DIR = runtime_cache_root(REPOSITORY_ROOT) / "notebook_12_pilot_inputs"

for directory in [
    AUDIT_DIR, PROCESSED_DIR, METADATA_DIR,
    PILOT_DIR, LOG_DIR, CACHE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PILOT_CANDIDATE = "Na_candidate_06"
CONFIG_FILE = ROOT / "configuration/pilot_dft_resource_config.yaml"
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).isoformat()

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("pymatgen:", getattr(pymatgen, "__version__", "unknown"))
print("Pilot candidate:", PILOT_CANDIDATE)
print("Output:", OUTPUT_ROOT)

In [ ]:
# Resolve Notebook 11 from the canonical clean-room namespace
NB14B = artifact_namespace("11", REPOSITORY_ROOT)


required = {
    "decision": NB14B / "metadata" / "11_final_decision.json",
    "output_manifest": NB14B / "metadata" / "11_output_file_manifest.csv",
    "case_manifest": NB14B / "processed" / "11_hypothesis_driven_dft_case_manifest.csv",
    "structure_manifest": NB14B / "processed" / "11_exact_structure_manifest.csv",
    "vasp_manifest": NB14B / "processed" / "11_vasp_input_template_manifest.csv",
    "pilot_order": NB14B / "processed" / "11_pilot_dft_execution_order.csv",
    "charged_structure": NB14B / "exact_structures" / PILOT_CANDIDATE / "charged" / "structure.json",
    "discharged_structure": NB14B / "exact_structures" / PILOT_CANDIDATE / "discharged" / "structure.json",
    "charged_inputs": NB14B / "dft_inputs" / PILOT_CANDIDATE / "charged",
    "discharged_inputs": NB14B / "dft_inputs" / PILOT_CANDIDATE / "discharged",
}

missing = [
    f"{role}: {path}" for role, path in required.items()
    if not path.exists()
]
if missing:
    raise FileNotFoundError(
        "Missing Notebook 11 inputs:\n" + "\n".join(missing)
    )

decision14b = json.loads(
    required["decision"].read_text(encoding="utf-8")
)
if decision14b.get("final_decision") != "FULL_GO_TO_PILOT_DFT":
    raise RuntimeError(
        f"Notebook 11 decision is not FULL_GO_TO_PILOT_DFT: "
        f"{decision14b.get('final_decision')}"
    )

manifest14b = pd.read_csv(required["output_manifest"])
manifest_failures = []

for row in manifest14b.itertuples(index=False):
    path = NB14B / str(row.relative_path)
    if not path.exists():
        manifest_failures.append({
            "relative_path": row.relative_path,
            "failure": "missing",
        })
        continue
    observed = sha256(path)
    if observed != str(row.sha256):
        manifest_failures.append({
            "relative_path": row.relative_path,
            "failure": "sha256_mismatch",
        })

manifest_audit = pd.DataFrame(manifest_failures)
if manifest_audit.empty:
    manifest_audit = pd.DataFrame([
        {"relative_path": "", "failure": "", "pass": True}
    ])
else:
    manifest_audit["pass"] = False

manifest_audit.to_csv(
    AUDIT_DIR / "12_notebook14B_manifest_verification.csv",
    index=False,
)

if len(manifest_failures):
    raise RuntimeError(
        "Notebook 11 output-manifest verification failed."
    )

print("Notebook 11 verified files:", len(manifest14b))

In [ ]:
# Load exact structures and calculate reaction balance

charged = Structure.from_dict(
    json.loads(required["charged_structure"].read_text(encoding="utf-8"))
)
discharged = Structure.from_dict(
    json.loads(required["discharged_structure"].read_text(encoding="utf-8"))
)

case_manifest = pd.read_csv(required["case_manifest"])
case_row = case_manifest[
    case_manifest["candidate_id"] == PILOT_CANDIDATE
].copy()

if len(case_row) != 1:
    raise RuntimeError(
        f"Expected one case-manifest row for {PILOT_CANDIDATE}; "
        f"found {len(case_row)}"
    )

case_row = case_row.iloc[0]

def host_composition(structure: Structure) -> Composition:
    amounts = {
        element: amount
        for element, amount in structure.composition.items()
        if element.symbol != "Na"
    }
    return Composition(amounts)

def host_reduced_and_factor(structure: Structure):
    reduced, factor = host_composition(
        structure
    ).get_reduced_composition_and_factor()
    return reduced, float(factor)

charged_host_reduced, charged_host_factor = host_reduced_and_factor(charged)
discharged_host_reduced, discharged_host_factor = host_reduced_and_factor(discharged)

host_match = charged_host_reduced.almost_equals(
    discharged_host_reduced,
    rtol=1e-8,
    atol=1e-10,
)

if not host_match:
    raise RuntimeError(
        "Charged and discharged structures do not reduce to the same non-Na host composition."
    )

charged_na_cell = float(charged.composition["Na"])
discharged_na_cell = float(discharged.composition["Na"])

charged_na_per_host_fu = charged_na_cell / charged_host_factor
discharged_na_per_host_fu = discharged_na_cell / discharged_host_factor
delta_na_per_host_fu = discharged_na_per_host_fu - charged_na_per_host_fu

if delta_na_per_host_fu <= 0:
    raise RuntimeError(
        "The discharged state must contain more Na per host formula unit."
    )

charged_volume_per_host_fu = charged.volume / charged_host_factor
discharged_volume_per_host_fu = discharged.volume / discharged_host_factor
signed_volume_change = (
    discharged_volume_per_host_fu - charged_volume_per_host_fu
) / charged_volume_per_host_fu
absolute_volume_change = abs(signed_volume_change)

balance = {
    "candidate_id": PILOT_CANDIDATE,
    "host_reduced_formula": charged_host_reduced.reduced_formula,
    "charged_cell_formula": charged.composition.reduced_formula,
    "discharged_cell_formula": discharged.composition.reduced_formula,
    "charged_cell_host_formula_units": charged_host_factor,
    "discharged_cell_host_formula_units": discharged_host_factor,
    "charged_na_per_host_formula_unit": charged_na_per_host_fu,
    "discharged_na_per_host_formula_unit": discharged_na_per_host_fu,
    "delta_na_per_host_formula_unit": delta_na_per_host_fu,
    "charged_energy_cell_normalization_factor": 1.0 / charged_host_factor,
    "discharged_energy_cell_normalization_factor": 1.0 / discharged_host_factor,
    "charged_volume_per_host_formula_unit": charged_volume_per_host_fu,
    "discharged_volume_per_host_formula_unit": discharged_volume_per_host_fu,
    "signed_volume_change_fraction_from_exact_input_structures": signed_volume_change,
    "absolute_volume_change_fraction_from_exact_input_structures": absolute_volume_change,
    "database_max_delta_volume": float(case_row["max_delta_volume"]),
    "balanced_reaction": (
        f"charged_host + {delta_na_per_host_fu:g} Na(metal) -> discharged_host"
    ),
    "voltage_formula": (
        "V = -[(E_discharged_cell / n_host_fu_discharged) "
        "- (E_charged_cell / n_host_fu_charged) "
        "- delta_Na * E_Na_atom] / delta_Na"
    ),
    "energy_unit_policy": (
        "Use final static energies in eV; the resulting eV per transferred Na is numerically equal to volts."
    ),
}

balance_df = pd.DataFrame([balance])
balance_df.to_csv(
    PROCESSED_DIR / "12_pilot_reaction_balance.csv",
    index=False,
)
(PROCESSED_DIR / "12_pilot_reaction_balance.json").write_text(
    json.dumps(balance, indent=2),
    encoding="utf-8",
)

display(balance_df.T)

In [ ]:
# Copy exact pilot inputs into a standalone private execution package

candidate_root = PILOT_DIR / PILOT_CANDIDATE
if candidate_root.exists():
    shutil.rmtree(candidate_root)

for state_role, source in [
    ("charged", required["charged_inputs"]),
    ("discharged", required["discharged_inputs"]),
]:
    target = candidate_root / state_role
    shutil.copytree(source, target)

    structure = charged if state_role == "charged" else discharged
    exact_dir = target / "00_exact_structure"
    exact_dir.mkdir(exist_ok=True)
    Poscar(structure).write_file(exact_dir / "POSCAR")
    (exact_dir / "structure.json").write_text(
        json.dumps(structure.as_dict(), indent=2),
        encoding="utf-8",
    )

for filename in [
    "11_hypothesis_driven_dft_case_manifest.csv",
    "11_exact_structure_manifest.csv",
    "11_vasp_input_template_manifest.csv",
]:
    source = NB14B / "processed" / filename
    if source.exists():
        shutil.copy2(source, PILOT_DIR / filename)

print("Standalone pilot candidate inputs copied.")

In [ ]:
# Prepare metallic bcc Na reference

NA_INITIAL_LATTICE_A = 4.23

na_structure = Structure.from_spacegroup(
    "Im-3m",
    Lattice.cubic(NA_INITIAL_LATTICE_A),
    ["Na"],
    [[0, 0, 0]],
)

na_root = PILOT_DIR / "Na_metal_reference"
relax_dir = na_root / "01_relax"
static_dir = na_root / "02_static_template"
relax_dir.mkdir(parents=True, exist_ok=True)
static_dir.mkdir(parents=True, exist_ok=True)

na_relax_settings = {
    "ENCUT": 520,
    "EDIFF": 1e-6,
    "EDIFFG": -0.01,
    "ISIF": 3,
    "NSW": 120,
    "ISPIN": 1,
    "ISMEAR": 1,
    "SIGMA": 0.20,
    "LASPH": True,
    "LREAL": False,
    "LWAVE": False,
    "LCHARG": False,
}

na_static_settings = {
    "ENCUT": 520,
    "EDIFF": 1e-7,
    "NSW": 0,
    "IBRION": -1,
    "ISPIN": 1,
    "ISMEAR": 1,
    "SIGMA": 0.05,
    "LASPH": True,
    "LREAL": False,
    "LWAVE": False,
    "LCHARG": False,
}

relax_set = MPRelaxSet(
    na_structure,
    force_gamma=True,
    user_incar_settings=na_relax_settings,
)
relax_set.write_input(
    relax_dir,
    potcar_spec=True,
)

static_set = MPStaticSet(
    na_structure,
    force_gamma=True,
    user_incar_settings=na_static_settings,
)
static_set.write_input(
    static_dir,
    potcar_spec=True,
)

Kpoints.gamma_automatic((12, 12, 12)).write_file(
    relax_dir / "KPOINTS"
)
Kpoints.gamma_automatic((18, 18, 18)).write_file(
    static_dir / "KPOINTS"
)

initial_static = static_dir / "POSCAR"
if initial_static.exists():
    initial_static.rename(
        static_dir / "POSCAR_INITIAL_REFERENCE"
    )

(static_dir / "README_STATIC_STAGE.txt").write_text(
    "After the Na-metal relaxation converges, copy ../01_relax/CONTCAR "
    "to this directory as POSCAR.\n"
    "Use the same PAW dataset and ENCUT policy as the electrode calculations.\n"
    "For the metallic reference, parse the reported sigma->0 energy when using finite smearing.\n",
    encoding="utf-8",
)

for path in PILOT_DIR.rglob("POTCAR"):
    if path.name == "POTCAR":
        path.unlink()

na_reference = {
    "initial_structure": "bcc Na, Im-3m",
    "initial_lattice_a_angstrom": NA_INITIAL_LATTICE_A,
    "n_atoms_in_reference_cell": len(na_structure),
    "energy_normalization": "E_Na_atom = E_Na_static_cell / n_atoms",
    "relax_kmesh": [12, 12, 12],
    "static_kmesh": [18, 18, 18],
    "potcar_policy": "Na_pv expected; verify against electrode POTCAR.spec",
}
(PROCESSED_DIR / "12_na_reference_definition.json").write_text(
    json.dumps(na_reference, indent=2),
    encoding="utf-8",
)

print("Na reference sites:", len(na_structure))

In [ ]:
# Resource configuration and scheduler templates

DEFAULT_CONFIG = {
    "resource_confirmed": False,
    "backend": "VASP",
    "scheduler": "SLURM",
    "account": "REPLACE_ME",
    "partition": "REPLACE_ME",
    "nodes": 1,
    "ntasks": 48,
    "cpus_per_task": 1,
    "walltime": "48:00:00",
    "module_commands": [
        "module purge",
        "module load REPLACE_WITH_SITE_VASP_MODULE",
    ],
    "vasp_command": "srun vasp_std",
    "vasp_version": "REPLACE_ME",
    "pseudopotential_library": "REPLACE_ME",
    "pseudopotential_version": "REPLACE_ME",
    "potcar_generation_method": "SITE_SPECIFIC_PRIVATE_METHOD",
    "notification_email": "",
}

if CONFIG_FILE.exists():
    config = yaml.safe_load(
        CONFIG_FILE.read_text(encoding="utf-8")
    )
else:
    config = DEFAULT_CONFIG.copy()
    template_path = OUTPUT_ROOT / "pilot_dft_resource_config_TEMPLATE.yaml"
    template_path.write_text(
        yaml.safe_dump(DEFAULT_CONFIG, sort_keys=False),
        encoding="utf-8",
    )

def contains_placeholder(value: Any) -> bool:
    text = json.dumps(value)
    return bool(
        re.search(
            r"REPLACE_ME|REPLACE_WITH|SITE_SPECIFIC_PRIVATE_METHOD",
            text,
        )
    )

required_config_fields = [
    "backend", "scheduler", "account", "partition",
    "nodes", "ntasks", "walltime",
    "module_commands", "vasp_command",
    "vasp_version", "pseudopotential_library",
    "pseudopotential_version", "potcar_generation_method",
]

resource_checks = []
for field in required_config_fields:
    value = config.get(field)
    resource_checks.append({
        "check": f"config_{field}",
        "pass": value not in [None, "", []] and not contains_placeholder(value),
        "value": json.dumps(value),
    })

resource_checks.append({
    "check": "resource_confirmed",
    "pass": bool(config.get("resource_confirmed") is True),
    "value": str(config.get("resource_confirmed")),
})

resource_audit = pd.DataFrame(resource_checks)
resource_audit.to_csv(
    AUDIT_DIR / "12_resource_configuration_audit.csv",
    index=False,
)

def slurm_script(job_name: str) -> str:
    modules = "\n".join(config.get("module_commands", []))
    email = str(config.get("notification_email", "")).strip()
    email_lines = (
        f"#SBATCH -mail-user={email}\n#SBATCH -mail-type=END,FAIL\n"
        if email else ""
    )
    return f'''#!/usr/bin/env bash
#SBATCH -job-name={job_name}
#SBATCH -account={config.get("account")}
#SBATCH -partition={config.get("partition")}
#SBATCH -nodes={config.get("nodes")}
#SBATCH -ntasks={config.get("ntasks")}
#SBATCH -cpus-per-task={config.get("cpus_per_task", 1)}
#SBATCH -time={config.get("walltime")}
{email_lines}
set -euo pipefail

{modules}

if [[ ! -f POTCAR ]]; then
    echo "ERROR: POTCAR is absent."
    echo "Generate POTCAR privately using the site-approved method and POTCAR.spec."
    echo "Never commit or distribute POTCAR."
    exit 2
fi

{config.get("vasp_command")} > vasp.stdout 2> vasp.stderr
'''

job_rows = []
for job_dir in [
    candidate_root / "charged" / "01_relax",
    candidate_root / "charged" / "02_static_template",
    candidate_root / "discharged" / "01_relax",
    candidate_root / "discharged" / "02_static_template",
    na_root / "01_relax",
    na_root / "02_static_template",
]:
    job_name = "pilot_" + "_".join(
        job_dir.relative_to(PILOT_DIR).parts
    )[:100]
    script = job_dir / "submit.slurm"
    script.write_text(
        slurm_script(job_name),
        encoding="utf-8",
    )
    script.chmod(
        script.stat().st_mode
        | stat.S_IXUSR
        | stat.S_IXGRP
    )
    job_rows.append({
        "job_directory": str(
            job_dir.relative_to(PILOT_DIR)
        ).replace("\\", "/"),
        "job_name": job_name,
        "scheduler_script": str(
            script.relative_to(PILOT_DIR)
        ).replace("\\", "/"),
        "stage": (
            "relax"
            if job_dir.name == "01_relax"
            else "static"
        ),
    })

job_manifest = pd.DataFrame(job_rows)
job_manifest.to_csv(
    PROCESSED_DIR / "12_pilot_job_manifest.csv",
    index=False,
)

display(resource_audit)

In [ ]:
# Execution order, static-stage helper and parser-ready schema

execution_steps = pd.DataFrame([
    {
        "step": 1,
        "job": "Na_metal_reference/01_relax",
        "action": "Generate private POTCAR, submit relaxation, verify convergence.",
    },
    {
        "step": 2,
        "job": f"{PILOT_CANDIDATE}/charged/01_relax",
        "action": "Generate private POTCAR, submit exact charged-state relaxation.",
    },
    {
        "step": 3,
        "job": f"{PILOT_CANDIDATE}/discharged/01_relax",
        "action": "Generate private POTCAR, submit exact discharged-state relaxation.",
    },
    {
        "step": 4,
        "job": "all 02_static_template directories",
        "action": "Only after relaxation convergence, copy each CONTCAR to the matching static directory as POSCAR.",
    },
    {
        "step": 5,
        "job": "Na_metal_reference/02_static_template",
        "action": "Submit the dense-mesh Na reference static calculation.",
    },
    {
        "step": 6,
        "job": f"{PILOT_CANDIDATE}/charged/02_static_template",
        "action": "Submit the charged-state static calculation.",
    },
    {
        "step": 7,
        "job": f"{PILOT_CANDIDATE}/discharged/02_static_template",
        "action": "Submit the discharged-state static calculation.",
    },
    {
        "step": 8,
        "job": "result collection",
        "action": "Preserve OUTCAR, vasprun.xml, OSZICAR, CONTCAR, INCAR, KPOINTS and calculation logs.",
    },
])
execution_steps.to_csv(
    PROCESSED_DIR / "12_pilot_execution_order.csv",
    index=False,
)

helper = PILOT_DIR / "prepare_static_from_relax.py"
helper.write_text(
    '''from pathlib import Path
import shutil

pairs = [
    (Path("Na_candidate_06/charged/01_relax/CONTCAR"),
     Path("Na_candidate_06/charged/02_static_template/POSCAR")),
    (Path("Na_candidate_06/discharged/01_relax/CONTCAR"),
     Path("Na_candidate_06/discharged/02_static_template/POSCAR")),
    (Path("Na_metal_reference/01_relax/CONTCAR"),
     Path("Na_metal_reference/02_static_template/POSCAR")),
]

for source, destination in pairs:
    if not source.exists() or source.stat().st_size == 0:
        raise FileNotFoundError(f"Missing converged relaxation structure: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print(f"Copied {source} -> {destination}")
''',
    encoding="utf-8",
)

result_schema = pd.DataFrame([
    {
        "calculation_role": "charged_static",
        "relative_directory": f"{PILOT_CANDIDATE}/charged/02_static_template",
        "required_files": "vasprun.xml|OUTCAR|OSZICAR|CONTCAR|INCAR|KPOINTS|POTCAR.spec",
        "energy_normalization": f"divide cell energy by {charged_host_factor:g}",
    },
    {
        "calculation_role": "discharged_static",
        "relative_directory": f"{PILOT_CANDIDATE}/discharged/02_static_template",
        "required_files": "vasprun.xml|OUTCAR|OSZICAR|CONTCAR|INCAR|KPOINTS|POTCAR.spec",
        "energy_normalization": f"divide cell energy by {discharged_host_factor:g}",
    },
    {
        "calculation_role": "na_reference_static",
        "relative_directory": "Na_metal_reference/02_static_template",
        "required_files": "vasprun.xml|OUTCAR|OSZICAR|CONTCAR|INCAR|KPOINTS|POTCAR.spec",
        "energy_normalization": f"divide cell energy by {len(na_structure)}",
    },
])
result_schema.to_csv(
    PROCESSED_DIR / "12_expected_pilot_result_schema.csv",
    index=False,
)

readme = f'''# Private pilot DFT package

Candidate: {PILOT_CANDIDATE}

This package contains exact charged/discharged structures traced to Notebook 01,
a metallic-bcc-Na reference, VASP-compatible input templates, reaction-balance
metadata and scheduler scripts.

## Critical rules

1. Generate POTCAR only in a private licensed computing environment.
2. Never commit, upload or distribute POTCAR.
3. Do not run a static job using POSCAR_INITIAL_REFERENCE.
4. Copy the converged relaxation CONTCAR to the static folder as POSCAR.
5. Use the same PAW-library version and correction policy for both endpoint states
   and the Na reference.
6. Preserve every convergence and failure log.
7. Do not interpret voltage until all three static calculations are converged.
8. The reaction uses {delta_na_per_host_fu:g} transferred Na per host formula unit.

The later parsing notebook will use:
- charged cell energy divided by {charged_host_factor:g}
- discharged cell energy divided by {discharged_host_factor:g}
- Na reference cell energy divided by {len(na_structure)}
'''
(PILOT_DIR / "README_PILOT_DFT.md").write_text(
    readme,
    encoding="utf-8",
)

display(execution_steps)
display(result_schema)

In [ ]:
# Final safety gates and private pilot ZIP

licensed_potcars = [
    path for path in PILOT_DIR.rglob("POTCAR")
    if path.name == "POTCAR"
]

potcar_specs = list(PILOT_DIR.rglob("POTCAR.spec"))
required_roles = {
    "charged_static",
    "discharged_static",
    "na_reference_static",
}

balance_audit = pd.DataFrame([
    {
        "check": "same_non_na_host_composition",
        "pass": bool(host_match),
    },
    {
        "check": "positive_delta_na",
        "pass": bool(delta_na_per_host_fu > 0),
    },
    {
        "check": "delta_na_matches_NaCoPCO7_to_Na3CoPCO7",
        "pass": bool(np.isclose(delta_na_per_host_fu, 2.0)),
    },
    {
        "check": "exact_input_volume_change_matches_database",
        "pass": bool(
            np.isclose(
                absolute_volume_change,
                float(case_row["max_delta_volume"]),
                rtol=1e-4,
                atol=1e-6,
            )
        ),
    },
])
balance_audit.to_csv(
    AUDIT_DIR / "12_reaction_balance_audit.csv",
    index=False,
)

gate_rows = [
    ("notebook14B_full_go_verified", True),
    ("notebook14B_manifest_verified", len(manifest_failures) == 0),
    ("pilot_exact_structures_loaded", len(charged) > 0 and len(discharged) > 0),
    ("reaction_balance_passed", bool(balance_audit["pass"].all())),
    ("na_reference_created", len(na_structure) > 0),
    ("all_required_job_directories_created", len(job_manifest) == 6),
    ("expected_result_roles_complete", set(result_schema["calculation_role"]) == required_roles),
    ("potcar_specs_present", len(potcar_specs) >= 6),
    ("no_licensed_potcar_present", len(licensed_potcars) == 0),
]
gates = pd.DataFrame(gate_rows, columns=["gate", "pass"])
gates.to_csv(
    AUDIT_DIR / "12_go_no_go_gate_audit.csv",
    index=False,
)

resource_ready = bool(resource_audit["pass"].all())

if not gates["pass"].all():
    decision = "HOLD_PILOT_PACKAGE_VALIDATION_FAILURE"
elif resource_ready:
    decision = "FULL_GO_SUBMIT_PILOT_DFT_JOBS"
else:
    decision = "PILOT_PACKAGE_READY_RESOURCE_CONFIRMATION_REQUIRED"

decision_payload = {
    "final_decision": decision,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "pilot_candidate": PILOT_CANDIDATE,
    "delta_na_per_host_formula_unit": delta_na_per_host_fu,
    "charged_host_formula_units_per_cell": charged_host_factor,
    "discharged_host_formula_units_per_cell": discharged_host_factor,
    "na_reference_atoms_per_cell": len(na_structure),
    "resource_configuration_complete": resource_ready,
    "dft_calculations_run": False,
    "licensed_potcar_written": False,
    "next_action": (
        "Transfer the private pilot package to the confirmed licensed HPC environment and submit relaxation jobs."
        if decision == "FULL_GO_SUBMIT_PILOT_DFT_JOBS"
        else "Complete and verify pilot_dft_resource_config.yaml before job submission."
    ),
}
(METADATA_DIR / "12_final_decision.json").write_text(
    json.dumps(decision_payload, indent=2),
    encoding="utf-8",
)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pymatgen": getattr(pymatgen, "__version__", "unknown"),
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
}
(METADATA_DIR / "12_software_environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

pilot_zip = OUTPUT_ROOT / "12_PRIVATE_PILOT_DFT_PACKAGE_NO_POTCAR.zip"
with zipfile.ZipFile(
    pilot_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(PILOT_DIR.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "POTCAR":
            continue
        archive.write(
            path,
            arcname=str(
                Path("pilot_dft_package")
                / path.relative_to(PILOT_DIR)
            ),
        )

manifest_rows = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file() and path.name != "12_output_file_manifest.csv":
        manifest_rows.append({
            "relative_path": str(path.relative_to(OUTPUT_ROOT)).replace("\\", "/"),
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        })
output_manifest = pd.DataFrame(manifest_rows)
output_manifest.to_csv(
    METADATA_DIR / "12_output_file_manifest.csv",
    index=False,
)

display(balance_audit)
display(gates)
print("FINAL DECISION:", decision)
print("Private pilot package:", pilot_zip)

if decision == "HOLD_PILOT_PACKAGE_VALIDATION_FAILURE":
    raise RuntimeError(decision)